# RO-070 Path following and tracking errors: every number in the Note

The car, the test course and the path coordinates live in `images/trackkit.py`: wheelbase 2.5 m, steering at most
26.6 deg, the kinematic bicycle model, commands at 20 Hz held 0.05 s, Euler steps of 0.01 s. The course: 10 m straight,
a 90 deg left arc of radius 10 m, 10 m straight; the car starts with its rear axle at (0, 1.0) heading 0 deg. The
driver is RO-069's line rule on the course's tangent at the nearest point. Nothing here is random.

In [1]:
import sys
sys.path.insert(0, 'images')
import numpy as np
import trackkit as k
deg = np.degrees

## 1. The opening: one line against the moving tangent

In [2]:
A = k.run(lambda q, t: k.first_line_rule(q), stop=lambda q: False, t_max=11.0)
i = 1100
p = k.course_point(k.nearest_s(A['x'][i], A['y'][i]))
print(f'one line: at {A["t"][i]:.2f} s rear axle ({A["x"][i]:.2f}, {A["y"][i]:.4f}), heading {deg(A["th"][i]):.2f} deg,'
      f' {np.hypot(A["x"][i] - p[0], A["y"][i] - p[1]):.2f} m from the course')
B = k.run(lambda q, t: k.line_rule(q))
print(f'tangent at the nearest point: reaches the end at {B["t"][-1]:.2f} s; course length {k.S_END:.3f} m')

one line: at 11.00 s rear axle (21.94, 0.0002), heading -0.01 deg, 5.57 m from the course
tangent at the nearest point: reaches the end at 18.15 s; course length 35.708 m


## 2. The bicycle model

In [3]:
print(f'largest steering angle {deg(k.DELTA_MAX):.2f} deg, tan = {np.tan(k.DELTA_MAX):.2f}, tightest circle {k.WHEELBASE / np.tan(k.DELTA_MAX):.2f} m')
for d in (k.DELTA_MAX, np.arctan(0.25)):
    print(f'delta {deg(d):.2f} deg: R = {k.WHEELBASE / np.tan(d):.2f} m, turn rate at 2 m/s {2 * np.tan(d) / k.WHEELBASE:.2f} rad/s,'
          f' after 8 s {8 * 2 * np.tan(d) / k.WHEELBASE:.2f} rad = {deg(8 * 2 * np.tan(d) / k.WHEELBASE):.0f} deg')
print(f'the arc: curvature {1 / k.R_ARC} per m; its direction turns {np.pi / 2:.4f} rad over {k.L_ARC:.3f} m = {np.pi / 2 / k.L_ARC:.3f} rad per m')
print(f'steering for the arc: tan delta = 2.5 x 0.1 = {2.5 * 0.1}, delta = {deg(np.arctan(0.25)):.2f} deg')
for v in (2.0, 5.0):
    print(f'lateral acceleration at {v} m/s on the 10 m arc: {v * v / 10:.2f} m/s^2; on the 5 m circle {v * v / 5:.2f}')
print(f'0.5 x mu x g with mu = 1.0: {0.5 * 1.0 * 9.81:.3f} m/s^2')
runs = {'one line': A, 'tangent': B,
        'held, following': k.run(lambda q, t: k.line_rule(q), hold=k.HOLD), 'held, tracking': k.run(k.tracker, hold=k.HOLD),
        'lateral only': k.run(lambda q, t: k.line_rule(q, k_h=0.0), t_max=25),
        'heading only': k.run(lambda q, t: k.line_rule(q, k_d=0.0), t_max=25)}
for name, r in runs.items():
    print(f'{name}: largest sideways acceleration v^2 tan(delta) / L = {np.max(np.abs(r["v"] ** 2 * np.tan(r["delta"]) / k.WHEELBASE)):.2f} m/s^2')

largest steering angle 26.57 deg, tan = 0.50, tightest circle 5.00 m
delta 26.57 deg: R = 5.00 m, turn rate at 2 m/s 0.40 rad/s, after 8 s 3.20 rad = 183 deg
delta 14.04 deg: R = 10.00 m, turn rate at 2 m/s 0.20 rad/s, after 8 s 1.60 rad = 92 deg
the arc: curvature 0.1 per m; its direction turns 1.5708 rad over 15.708 m = 0.100 rad per m
steering for the arc: tan delta = 2.5 x 0.1 = 0.25, delta = 14.04 deg
lateral acceleration at 2.0 m/s on the 10 m arc: 0.40 m/s^2; on the 5 m circle 0.80
lateral acceleration at 5.0 m/s on the 10 m arc: 2.50 m/s^2; on the 5 m circle 5.00
0.5 x mu x g with mu = 1.0: 4.905 m/s^2


one line: largest sideways acceleration v^2 tan(delta) / L = 0.80 m/s^2
tangent: largest sideways acceleration v^2 tan(delta) / L = 0.80 m/s^2
held, following: largest sideways acceleration v^2 tan(delta) / L = 0.80 m/s^2
held, tracking: largest sideways acceleration v^2 tan(delta) / L = 2.50 m/s^2
lateral only: largest sideways acceleration v^2 tan(delta) / L = 0.80 m/s^2
heading only: largest sideways acceleration v^2 tan(delta) / L = 0.42 m/s^2


## 3. Path following against trajectory tracking (both held 1.5 s)

In [4]:
PF = k.run(lambda q, t: k.line_rule(q), hold=k.HOLD)
TR = k.run(k.tracker, hold=k.HOLD)
at_pf = np.array([k.timetable_s(t) for t in PF['t']]) - PF['s']
at_tr = np.array([k.timetable_s(t) for t in TR['t']]) - TR['s']
print(f'timetable reaches the end at {k.S_END / 2:.2f} s')
print(f'at 1.5 s the timetable is {at_tr[150]:.2f} m ahead; tracker speed {k.V_BASE + k.K_S * at_tr[150]:.1f} m/s')
j = np.where(np.abs(at_tr) > 0.1)[0][-1] + 1
print(f'tracker: along-track error within 0.1 m from {TR["t"][j]:.2f} s; top speed {TR["v"].max():.1f} m/s; end at {TR["t"][-1]:.2f} s,'
      f' along-track error then {at_tr[-1]:.3f} m')
print(f'follower: end at {PF["t"][-1]:.2f} s, along-track error then {at_pf[-1]:.2f} m')
print('follower lateral error identical to the run without the hold:', np.max(np.abs(PF['e'][150:150 + len(B['e']) - 1] - B['e'][:-1])) == 0.0)

timetable reaches the end at 17.85 s
at 1.5 s the timetable is 3.00 m ahead; tracker speed 5.0 m/s
tracker: along-track error within 0.1 m from 4.89 s; top speed 5.0 m/s; end at 17.90 s, along-track error then 0.092 m
follower: end at 19.65 s, along-track error then 3.59 m
follower lateral error identical to the run without the hold: True


## 4. Path coordinates by hand: the car at (14.0, 0.0) heading 10 deg

In [5]:
q = (14.0, 0.0, np.radians(10))
dist = np.hypot(q[0] - 10, q[1] - 10)
phi = np.arctan2(q[1] - 10, q[0] - 10)
P = (10 + 10 * (q[0] - 10) / dist, 10 + 10 * (q[1] - 10) / dist)
print(f'distance from the centre {dist:.3f} m; P = ({P[0]:.3f}, {P[1]:.3f}); angle from the centre {deg(phi):.2f} deg')
print(f'swept since the arc began: {phi + np.pi / 2:.4f} rad = {deg(phi + np.pi / 2):.2f} deg; s = 10 + 10 x {phi + np.pi / 2:.4f} = {10 + 10 * (phi + np.pi / 2):.3f} m')
thp = phi + np.pi / 2
dx, dy = q[0] - P[0], q[1] - P[1]
print(f'course heading at P {deg(thp):.2f} deg; dx {dx:.3f}, dy {dy:.3f}')
print(f'-sin x dx = {-np.sin(thp) * dx:.4f}, cos x dy = {np.cos(thp) * dy:.4f}, e = {-np.sin(thp) * dx + np.cos(thp) * dy:.4f}; 10 - dist = {10 - dist:.4f}')
print(f'heading error {10 - deg(thp):.2f} deg; trackkit: {k.frenet(*q)[0]:.3f}, {k.frenet(*q)[1]:.4f}, {deg(k.frenet(*q)[2]):.2f}')

distance from the centre 10.770 m; P = (13.714, 0.715); angle from the centre -68.20 deg
swept since the arc began: 0.3805 rad = 21.80 deg; s = 10 + 10 x 0.3805 = 13.805 m
course heading at P 21.80 deg; dx 0.286, dy -0.715
-sin x dx = -0.1063, cos x dy = -0.6641, e = -0.7703; 10 - dist = -0.7703
heading error -11.80 deg; trackkit: 13.805, -0.7703, -11.80


How the errors change at that pose, at 2 m/s with the arc's steering, 14.04 deg

In [6]:
v, the, e, kap = 2.0, np.radians(10) - thp, 10 - dist, 0.1
print(f'v cos th_e = {v * np.cos(the):.4f}; angular rate about the centre {v * np.cos(the) / (10 - e):.5f}; s dot = {v * np.cos(the) / (1 - kap * e):.4f}')
print(f'e dot = {v * np.sin(the):.4f} m/s')
print(f'theta dot = {v * 0.25 / 2.5:.4f}; course turns at {kap * v * np.cos(the) / (1 - kap * e):.4f}; th_e dot = {v * 0.25 / 2.5 - kap * v * np.cos(the) / (1 - kap * e):.4f}')

v cos th_e = 1.9577; angular rate about the centre 0.18177; s dot = 1.8177
e dot = -0.4090 m/s
theta dot = 0.2000; course turns at 0.1818; th_e dot = 0.0182


## 5. The two errors, the two axles, and the run's numbers

In [7]:
for name, kd, kh in (('both', 0.5, 2.0), ('lateral only', 0.5, 0.0), ('heading only', 0.0, 2.0)):
    r = k.run(lambda q, t: k.line_rule(q, k_d=kd, k_h=kh), t_max=25)
    zc = r['s'][np.where(np.diff(np.sign(r['e'])) != 0)[0]]
    print(f'{name}: peak |e| {np.abs(r["e"]).max():.3f} m; e at the end of the straight {np.interp(10, r["s"], r["e"]):.3f} m;'
          f' end of course {r["e"][-1]:.3f} m; crosses the course at s = {np.round(zc, 2)}')
q1 = (B['x'][100], B['y'][100], B['th'][100])
ef = k.frenet(*k.front_axle(q1), q1[2])[1]
print(f'at 1.0 s: rear e {B["e"][100]:.4f} m, heading error {deg(B["the"][100]):.2f} deg; front axle e {ef:.4f} m;'
      f' e + 2.5 sin(th_e) = {B["e"][100] + 2.5 * np.sin(B["the"][100]):.4f}')
i_in = int(np.argmax(np.abs(B['e']) < 0.1))
arc = (B['s'] > k.L1 + 5) & (B['s'] <= k.L1 + k.L_ARC)
j = np.where((B['s'] > k.L1 + k.L_ARC) & (np.abs(B['e']) > 0.1))[0][-1]
print(f'peak |e| {np.abs(B["e"]).max():.3f} m; first within 0.1 m at s = {B["s"][i_in]:.2f} m ({B["t"][i_in]:.2f} s);'
      f' on the arc {B["e"][arc].min():.3f} m at s = {B["s"][arc][np.argmin(B["e"][arc])]:.2f} m; back within 0.1 m from s = {B["s"][j]:.2f} m')
print(f'estimate: the arc needs {2 * 0.1:.2f} rad/s; -0.5 e = 0.2 gives e = {-0.2 / 0.5:.2f} m; '
      f'with the outside factor: course turns at {0.1 * 2 / (1 - 0.1 * B["e"][arc].min()):.4f} rad/s')
print(f'steering on the arc at s = 20 m: {deg(np.interp(20, B["s"], B["delta"])):.2f} deg (the arc alone needs 14.04)')

both: peak |e| 1.000 m; e at the end of the straight 0.039 m; end of course -0.015 m; crosses the course at s = [10.82]
lateral only: peak |e| 2.092 m; e at the end of the straight 0.216 m; end of course -0.775 m; crosses the course at s = [ 3.16  9.62 14.56 24.5  30.79]
heading only: peak |e| 1.000 m; e at the end of the straight 1.000 m; end of course -0.584 m; crosses the course at s = [20.86]
at 1.0 s: rear e 0.7373 m, heading error -10.74 deg; front axle e 0.2715 m; e + 2.5 sin(th_e) = 0.2715


peak |e| 1.000 m; first within 0.1 m at s = 7.70 m (3.88 s); on the arc -0.388 m at s = 25.69 m; back within 0.1 m from s = 30.92 m
estimate: the arc needs 0.20 rad/s; -0.5 e = 0.2 gives e = -0.40 m; with the outside factor: course turns at 0.1925 rad/s
steering on the arc at s = 20 m: 13.85 deg (the arc alone needs 14.04)
